# CTU-Chat Remote Reranker on Colab

Run every cell in order on a GPU runtime. The notebook loads `BAAI/bge-reranker-v2-m3`, serves batched scores through FastAPI, and exposes the service through a temporary Cloudflare Quick Tunnel. Use it for experiments, not permanent production hosting.

In [ ]:
!pip -q install -U FlagEmbedding fastapi uvicorn httpx
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import getpass
import secrets
import threading
import torch
from FlagEmbedding import FlagReranker

MODEL_NAME = 'BAAI/bge-reranker-v2-m3'
MAX_DOCUMENTS = 64
MAX_TEXT_CHARS = 8000
API_KEY = getpass.getpass('Create an API key for this Colab session: ').strip()
if len(API_KEY) < 16:
    raise ValueError('Use an API key with at least 16 characters')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (switch Colab runtime to GPU)')
reranker = FlagReranker(MODEL_NAME, use_fp16=torch.cuda.is_available())
inference_lock = threading.Lock()
print('Model ready:', MODEL_NAME)

In [ ]:
from typing import Annotated
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title='CTU Remote Reranker', version='1.0')

class Candidate(BaseModel):
    id: str = Field(min_length=1, max_length=128)
    text: str = Field(max_length=MAX_TEXT_CHARS)

class RerankRequest(BaseModel):
    query: str = Field(min_length=1, max_length=4000)
    documents: list[Candidate] = Field(min_length=1, max_length=MAX_DOCUMENTS)

def authorize(authorization: str | None) -> None:
    expected = f'Bearer {API_KEY}'
    if authorization is None or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='Unauthorized')

@app.get('/health')
def health(authorization: Annotated[str | None, Header()] = None):
    authorize(authorization)
    return {
        'status': 'ok',
        'model': MODEL_NAME,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'max_documents': MAX_DOCUMENTS,
    }

@app.post('/rerank')
def rerank(request: RerankRequest, authorization: Annotated[str | None, Header()] = None):
    authorize(authorization)
    ids = [document.id for document in request.documents]
    if len(ids) != len(set(ids)):
        raise HTTPException(status_code=422, detail='Document IDs must be unique')
    pairs = [[request.query, document.text] for document in request.documents]
    with inference_lock:
        raw_scores = reranker.compute_score(pairs, normalize=False)
    if hasattr(raw_scores, 'tolist'):
        raw_scores = raw_scores.tolist()
    if not isinstance(raw_scores, (list, tuple)):
        raw_scores = [raw_scores]
    if len(raw_scores) != len(request.documents):
        raise HTTPException(status_code=500, detail='Model returned an invalid score count')
    return {
        'model': MODEL_NAME,
        'scores': [
            {'id': document.id, 'score': float(score)}
            for document, score in zip(request.documents, raw_scores)
        ],
    }

In [ ]:
import time
import uvicorn
import httpx

def run_api():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='info')

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
headers = {'Authorization': f'Bearer {API_KEY}'}
for _ in range(60):
    try:
        response = httpx.get('http://127.0.0.1:8000/health', headers=headers, timeout=2)
        response.raise_for_status()
        print(response.json())
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('FastAPI did not start')

In [ ]:
import pathlib
import re
import subprocess

tunnel_log_path = pathlib.Path('/tmp/ctu-cloudflared.log')
tunnel_log_handle = tunnel_log_path.open('w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)
PUBLIC_URL = None
for _ in range(120):
    time.sleep(0.5)
    log_text = tunnel_log_path.read_text(errors='replace') if tunnel_log_path.exists() else ''
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break
if PUBLIC_URL is None:
    raise RuntimeError('Cloudflare tunnel did not publish a URL. Log:\n' + log_text[-2000:])
print('Public URL:', PUBLIC_URL)
print('\nPut these values in your local .env:')
print('RAG_RERANKER_BACKEND=remote')
print(f'RAG_REMOTE_RERANKER_URL={PUBLIC_URL}')
print('RAG_REMOTE_RERANKER_API_KEY=<the same key entered above>')

In [ ]:
payload = {
    'query': 'Lệ phí xin cấp bản sao văn bằng tốt nghiệp là bao nhiêu?',
    'documents': [
        {'id': 'diploma', 'text': 'Phiếu đề nghị cấp bản sao văn bằng. Lệ phí cấp một bản sao là 10.000 đồng.'},
        {'id': 'tuition', 'text': 'Bảng mức thu học phí chương trình đại học chính quy khóa 52.'},
    ],
}
response = httpx.post(f'{PUBLIC_URL}/rerank', headers=headers, json=payload, timeout=60)
response.raise_for_status()
print(response.json())